# Quality Spec Analysis: Within Spec vs Off Spec Batches

**Objective:** Compare sensor behavior between batches that met quality specifications vs those that didn't, to identify which process parameters most strongly predict quality outcomes.

**Data Source:** Reaction-phase analysis data from the manufacturing historian system.

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Load anonymized data
df = pd.read_excel('../data/raw/sample_anonymized_data.xlsx', engine='openpyxl')

# The Excel file has data starting from row 0 with headers already applied
# Assign clean column names based on the sensor mapping
df.columns = [
    'Record_ID', 'Timestamp', 'RPM', 'Jacket_Water_Temp_C', 'Temp_Non_Drive_C',
    'Temp_Drive_C', 'Initiator_Flow_Rate', 'Initiator_Totalizer', 'TFE_Flow_Rate',
    'TFE_Totalizer', 'HFP_Flow_Rate', 'HFP_Totalizer', 'Ethane_Flow_Rate',
    'Ethane_Totalizer', 'PPVE_Flow_Rate', 'PPVE_Totalizer', 'DI_Water_Flow_Rate',
    'DI_Water_Totalizer', 'TFE_Pressure_Bar', 'Stage_Primary', 'Stage_Secondary',
    'Quality_Report'
][:len(df.columns)]  # Adjust if column count differs

# Convert numerics
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(f'Loaded {len(df):,} records')
print(f'Columns: {len(df.columns)}')
df.head()

## 2. Quality Distribution

Examine the breakdown of Within Spec vs Off Spec batches and the types of specification failures.

In [ ]:
# Check what quality categories exist
if 'Quality_Report' in df.columns:
    print('Quality Report categories:')
    print(df['Quality_Report'].value_counts())
else:
    # Try the last few columns which might contain quality info
    for col in df.columns[-3:]:
        unique_vals = df[col].dropna().unique()
        if len(unique_vals) < 20:
            print(f'{col}: {unique_vals}')

## 3. Sensor Comparison: Within Spec vs Off Spec

Compare temperature, pressure, and TFE quantity distributions between quality outcomes to identify which parameters differentiate good vs bad batches.

In [ ]:
# Key sensor parameters to compare
analysis_params = ['TFE_Totalizer', 'Temp_Drive_C', 'TFE_Pressure_Bar']

# Filter available numeric columns
available_params = [col for col in analysis_params if col in df.columns]

if available_params:
    fig, axes = plt.subplots(1, len(available_params), figsize=(6*len(available_params), 5))
    if len(available_params) == 1:
        axes = [axes]
    
    quality_col = 'Quality_Report' if 'Quality_Report' in df.columns else df.columns[-1]
    
    for i, param in enumerate(available_params):
        sns.boxplot(x=quality_col, y=param, data=df, ax=axes[i], palette='Set2')
        axes[i].set_title(f'{param} by Quality Outcome', fontweight='bold')
        axes[i].tick_params(axis='x', rotation=30)
    
    plt.tight_layout()
    plt.savefig('../dashboards/quality_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: dashboards/quality_comparison.png')
else:
    print('Analysis parameters not found in columns. Check column names.')

In [ ]:
# Temperature & Pressure distribution: Within Spec vs Off Spec (KDE plots)
quality_col = 'Quality_Report' if 'Quality_Report' in df.columns else df.columns[-1]
temp_col = 'Temp_Drive_C' if 'Temp_Drive_C' in df.columns else None
pressure_col = 'TFE_Pressure_Bar' if 'TFE_Pressure_Bar' in df.columns else None

if temp_col and pressure_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for category in df[quality_col].dropna().unique():
        subset = df[df[quality_col] == category]
        if len(subset) > 10:
            subset[temp_col].dropna().plot.kde(ax=axes[0], label=category, alpha=0.7)
            subset[pressure_col].dropna().plot.kde(ax=axes[1], label=category, alpha=0.7)
    
    axes[0].set_title('Temperature Distribution by Quality', fontweight='bold')
    axes[0].set_xlabel('Temperature (°C)')
    axes[0].legend()
    
    axes[1].set_title('Pressure Distribution by Quality', fontweight='bold')
    axes[1].set_xlabel('Pressure (bar)')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('../dashboards/quality_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: dashboards/quality_distributions.png')

## 4. Statistical Significance

Test whether the differences in sensor readings between quality categories are statistically significant.

In [ ]:
# Summary statistics by quality category
quality_col = 'Quality_Report' if 'Quality_Report' in df.columns else df.columns[-1]
numeric_analysis_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != 'Record_ID']

summary = df.groupby(quality_col)[numeric_analysis_cols[:6]].agg(['mean', 'std', 'min', 'max']).round(3)
print('Summary statistics by quality category:')
summary

## 5. Key Findings

The analysis reveals which process parameters most strongly differentiate Within Spec from Off Spec batches, providing actionable targets for process optimization and predictive quality monitoring.

**Next steps:** Feed these insights into the Power BI dashboard as threshold indicators and alert conditions.